In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [2]:
import json
import os
with open('chroma_inputs.jsonl', 'r') as f:
    lines = f.readlines()
    chroma_inputs = [json.loads(line) for line in lines if line.strip()]

In [3]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chromadb")

In [4]:
chroma_client.heartbeat()

1763008013359404900

In [5]:
import os
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

collection = chroma_client.create_collection(
    name = "my_collection",
    metadata={"description": "My embeddings collection"},
    embedding_function = OpenAIEmbeddingFunction(
        api_key = os.getenv("OPENAI_API_KEY"),
        model_name="text-embedding-3-small")
)

In [6]:
# Add data in batches
batch_size = 5000  # Safe batch size (under the 5461 ChromaDB's limit)
total_items = len(chroma_inputs)

for i in range(0, total_items, batch_size):
    batch = chroma_inputs[i:i + batch_size]
    
    collection.add(
        ids=[item['id'] for item in batch],
        documents=[item['text'] for item in batch],
        embeddings=[item['embedding'] for item in batch]
    )
    
    print(f"Added batch {i//batch_size + 1}: {len(batch)} documents (total: {min(i + batch_size, total_items)}/{total_items})")

print(f"\nCompleted! Collection now has {collection.count()} documents")

Added batch 1: 5000 documents (total: 5000/63427)
Added batch 2: 5000 documents (total: 10000/63427)
Added batch 3: 5000 documents (total: 15000/63427)
Added batch 4: 5000 documents (total: 20000/63427)
Added batch 5: 5000 documents (total: 25000/63427)
Added batch 6: 5000 documents (total: 30000/63427)
Added batch 7: 5000 documents (total: 35000/63427)
Added batch 8: 5000 documents (total: 40000/63427)
Added batch 9: 5000 documents (total: 45000/63427)
Added batch 10: 5000 documents (total: 50000/63427)
Added batch 11: 5000 documents (total: 55000/63427)
Added batch 12: 5000 documents (total: 60000/63427)
Added batch 13: 3427 documents (total: 63427/63427)

Completed! Collection now has 63427 documents


In [7]:
print(f"Added {len(chroma_inputs)} documents to ChromaDB")
print(f"Collection now has {collection.count()} documents")

Added 63427 documents to ChromaDB
Collection now has 63427 documents


With the embedding function, we can now perform the query:

In [ ]:
collection.query(
    query_texts = ["What was the revenue of Cemex in 2023?"], 
    n_results = 10
)
# It does not work, inapropiate table chunking

{'ids': [['16963_65767_18224',
   '15950_65685_3128',
   '16458_65741_15104',
   '16403_65734_15180',
   '15918_65681_4161',
   '16115_65704_5159',
   '16062_65699_1538',
   '15759_65671_5194',
   '15276_65608_5713',
   '16825_65758_17297']],
 'embeddings': None,
 'documents': [['second quarter 2025 results conference call and audio webcast presentation include statistical data regarding the production, distribution, marketing and sale of cement,\nready-mix\nconcrete, clinker,\naggregates, and Urbanization Solutions. Cemex generated some of this data internally, and some was obtained from independent industry publications and reports that Cemex believes to be reliable sources. Cemex has not independently verified this data\nnor sought the consent of any organization to refer to their reports in the reports, presentations, and documents to be disclosed in the events referenced herein. Cemex acts in strict compliance of antitrust laws and as such, among other measures,\nmaintains an inde